# Retrain the MedGuard PHI model on MIMIC-III

Fine-tunes `obi/deid_roberta_i2b2` on MIMIC-III notes whose de-identification tags are replaced with fake values, then compares it with the original model on held-out patients and held-out names.

**Set the runtime first:** Runtime > Change runtime type > GPU.

**MIMIC data rules**
- Check that this use fits your PhysioNet data use agreement. The default `bigquery` source reads PhysioNet's hosted copy under your own credentials. It needs a Google account linked to PhysioNet with BigQuery access enabled.
- Do not print or display note text, and do not save this notebook with note text in its output.
- The trained weights derive from MIMIC. Do not publish them or commit them.

In [ ]:
# Settings
PROJECT_ID = ""  # Your Google Cloud project, used for BigQuery billing and quota
DATA_SOURCE = "bigquery"  # "bigquery" or "drive"
BQ_TABLE = "physionet-data.mimiciii_notes.noteevents"
DRIVE_FILE = "/content/drive/MyDrive/NOTEEVENTS.csv.gz"  # Only used when DATA_SOURCE is "drive"
CATEGORIES = ["Nursing/other", "Radiology", "Nursing", "Physician", "Discharge summary"]

PER_CATEGORY = 1000  # Training notes per category. Raise this after the pilot works.
EPOCHS = 2
BATCH_SIZE = 16
EVAL_PER_CATEGORY = 200
OUTPUT_DIR = "models/pilot"
SAVE_TO_DRIVE = False  # Leave off unless your data use agreement allows storing the weights in Drive

REPO_URL = "https://github.com/azaveri879/medguard-llm.git"
REPO_BRANCH = "add-gateway-and-eval"
NOTES_CSV = "/content/notes.csv"

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found. Set Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0))

!pip install -q faker transformers==5.22.2
![ -d /content/medguard-llm ] || git clone -q --branch {REPO_BRANCH} {REPO_URL} /content/medguard-llm
%cd /content/medguard-llm/eval

In [ ]:
# Writes notes to this runtime's temporary disk. Prints a row count only.
import csv
import gzip
import shutil

csv.field_size_limit(10 ** 9)

if DATA_SOURCE == "bigquery":
    assert PROJECT_ID, "Set PROJECT_ID in the settings cell."
    from google.cloud import bigquery
    from google.colab import auth

    auth.authenticate_user()
    client = bigquery.Client(project=PROJECT_ID)
    query = f"""
        SELECT subject_id, TRIM(category) AS category, iserror, text
        FROM `{BQ_TABLE}`
        WHERE TRIM(category) IN UNNEST(@categories) AND text LIKE '%[**%'
    """
    config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("categories", "STRING", CATEGORIES)]
    )
    rows = 0
    with open(NOTES_CSV, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["subject_id", "category", "iserror", "text"])
        for row in client.query(query, job_config=config).result(page_size=5000):
            writer.writerow([row.subject_id, row.category, row.iserror or "", row.text])
            rows += 1
else:
    from google.colab import drive

    drive.mount("/content/drive")
    with gzip.open(DRIVE_FILE, "rb") as source, open(NOTES_CSV, "wb") as target:
        shutil.copyfileobj(source, target)
    with open(NOTES_CSV, newline="", encoding="utf-8", errors="replace") as handle:
        rows = sum(1 for _ in csv.reader(handle)) - 1

print(f"Notes available: {rows:,}")

In [ ]:
# Training uses only patients in the train split and names from the train list.
!python train_deid.py --output-dir {OUTPUT_DIR} --per-category {PER_CATEGORY} --epochs {EPOCHS} --batch-size {BATCH_SIZE} < {NOTES_CSV}

In [ ]:
# Compare both models on the same held-out patients and held-out names. Output is aggregate only.
from pathlib import Path

models = {"original": "obi/deid_roberta_i2b2", "retrained": str(Path(OUTPUT_DIR).resolve())}
for label, model in models.items():
    print(f"\n=== {label}")
    !PHI_NER_MODEL={model} python deid_benchmark.py --mimic --split test --faker-names --per-category {EVAL_PER_CATEGORY} --detectors pipeline,known-patient-care --output results/deid_colab_{label}.md < {NOTES_CSV}

In [ ]:
# The runtime's disk is erased when the session ends, including the notes and weights.
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/medguard-models
    !cp -r {OUTPUT_DIR} /content/drive/MyDrive/medguard-models/
    print("Saved model to Drive")
else:
    print("Model not saved. Download only the aggregate tables from eval/results if you want to keep them.")